# Session 15 · Case study: a RAG classification assistant and its evaluation

**Block 2 practice.** Build a RAG prototype that proposes a heading for a new description of goods from similar past decisions, with cited BTI references, and evaluate retrieval with **heading hit@5** on ten labelled requests (and on 500 more, because ten are too few). Then compare retrieval variants and check the answers of the prototype.

A request is a decision of 2022–2023 whose heading customs decided; a retrieved decision is relevant if it has the same heading. The language model can only propose a heading that a retrieved decision supports, so heading hit@k is an upper bound on the accuracy of the assistant.

**LLM access.** The prototype in section 5 calls an OpenAI-compatible endpoint (`LLM_BASE_URL`, `LLM_MODEL`, `LLM_API_KEY`; default Ollama with `llama3.2`). Without a reachable endpoint, section 5 is skipped; everything else runs offline. Run time: about 5 minutes with a GPU (the chunk-size comparison re-embeds the collection twice).

Theory: [02-evaluating-and-improving-rag.md](../theory/02-evaluating-and-improving-rag.md). The functions come from the workspace package `bti_assistant` (`../workspace/src`).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
sys.path.insert(0, str(ROOT / "sessions" / "15-rag-and-agents" / "workspace" / "src"))   # the workspace package

decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet")
heading_texts = dict(zip(nomenclature["heading"], nomenclature["heading_description"]))
year = decisions["start_date"].dt.year
docs = decisions[year <= 2021].sample(10_000, random_state=0).reset_index(drop=True)      # the collection
requests = decisions[year >= 2022].sample(2_000, random_state=0).reset_index(drop=True)   # new requests
print(len(docs), "decisions in the collection,", len(requests), "requests from 2022-2023")

## 1. Ten labelled requests (and 500 more)

In [ ]:
languages = ["de", "fr", "en", "nl", "pl", "cs", "es", "it", "sv", "da"]
ten = requests.groupby("language").head(1)
ten = ten[ten["language"].isin(languages)].reset_index(drop=True)
evalset = requests.iloc[:500]
show = ten.assign(heading_text=ten["heading"].map(heading_texts).str[:50],
                  description=ten["description"].str.replace(r"\s+", " ", regex=True).str[:70])
show[["language", "heading", "heading_text", "description"]]

## 2. Retrievers: dense, keyword, hybrid

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-small"
model = SentenceTransformer(MODEL_NAME)
cache = Path("embeddings_cache")
cache.mkdir(exist_ok=True)


def embed(name, texts, prefix):
    """Encode once, then load from disk; the file name records model, prefix and content."""
    path = cache / f"{MODEL_NAME.split('/')[-1]}_{prefix.strip(': ')}_{name}.npy"
    if path.exists():
        return np.load(path)
    E = model.encode([prefix + t for t in texts], batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    np.save(path, E)
    return E


from bti_assistant.chunking import chunk_documents
from bti_assistant.keyword import BM25
from bti_assistant.hybrid import reciprocal_rank_fusion
from bti_assistant.metrics import evaluate_heading_retrieval

H = dict(zip(docs["bti_reference"], docs["heading"]))
E = embed("collection_docs", docs["description"].tolist(), "passage: ")     # one vector per description
Q_ten = embed("ten_requests", ten["description"].tolist(), "query: ")
Q_500 = embed("500_requests", evalset["description"].tolist(), "query: ")
refs = docs["bti_reference"].to_numpy()
bm25 = BM25(list(refs), docs["description"].tolist())


def dense(qvec, depth=50):
    return list(refs[np.argsort(-(E @ qvec))[:depth]])


def keyword(text, depth=50):
    return [ref for ref, _ in bm25.search(text, k=depth)]


def evaluate(rankings, reqs, k=5):
    truth = {i: h for i, h in enumerate(reqs["heading"])}
    return evaluate_heading_retrieval(truth, {i: [H[r] for r in rank] for i, rank in enumerate(rankings)}, k=k)


rank = {}
for name, reqs, Q in [("ten", ten, Q_ten), ("500", evalset, Q_500)]:
    d = [dense(q) for q in Q]
    kw = [keyword(t) for t in reqs["description"]]                # plain-Python BM25: about a minute for 500
    rank[name] = {"dense": d, "keyword": kw,
                  "hybrid": [reciprocal_rank_fusion([a, b]) for a, b in zip(d, kw)]}
results = pd.DataFrame({(name, method): evaluate(r, ten if name == "ten" else evalset)
                        for name, methods in rank.items() for method, r in methods.items()}).T
results.round(3)

**Question.** How different are the scores on ten and on 500 requests? Which ranking of the three methods would you report from the ten alone?

## 3. Per-request results on the ten requests

In [ ]:
per_request = pd.DataFrame({
    "language": ten["language"], "true": ten["heading"],
    "top5 headings (hybrid)": [[H[r] for r in rank["ten"]["hybrid"][i][:5]] for i in range(len(ten))],
})
per_request["hit@5"] = [t in h for t, h in zip(per_request["true"], per_request["top5 headings (hybrid)"])]
per_request

## 4. Variants: chunk size, chapter filter, similarity vote

In [ ]:
rows = []
for size in [40, 120]:
    ch = chunk_documents(dict(zip(docs["bti_reference"], docs["description"])), size=size, overlap=size // 6)
    Ec = embed(f"collection_{size}w", [c.text for c in ch], "passage: ")
    doc_of = np.array([c.doc_id for c in ch])
    rankings = []
    for q in Q_500:
        order = doc_of[np.argsort(-(Ec @ q))]
        rankings.append(list(dict.fromkeys(order))[:50])        # best chunk per decision
    rows.append({"variant": f"dense, {size}-word chunks", **evaluate(rankings, evalset)})

chapter = docs["heading"].str[:2].to_numpy()
filtered = [list(refs[np.argsort(-np.where(chapter == t[:2], E @ q, -np.inf))[:50]])
            for q, t in zip(Q_500, evalset["heading"])]
rows.append({"variant": "dense + chapter filter (chapter known)", **evaluate(filtered, evalset)})
rows.append({"variant": "dense, one vector per description", **evaluate(rank["500"]["dense"], evalset)})
rows.append({"variant": "hybrid (RRF)", **evaluate(rank["500"]["hybrid"], evalset)})
pd.DataFrame(rows).round(3)

In [ ]:
from bti_assistant.rag import vote_headings
from bti_assistant.store import Hit


def vote_accuracy(k):
    correct = []
    for q, t in zip(Q_500, evalset["heading"]):
        idx = np.argsort(-(E @ q))[:k]
        hits = [Hit(refs[i], refs[i], "", float(E[i] @ q), H[refs[i]]) for i in idx]
        correct.append(vote_headings(hits)[0] == t)
    return np.mean(correct)


print("retrieval-only baseline: heading of the similarity-weighted vote")
print({k: round(vote_accuracy(k), 3) for k in [1, 5, 10]})

The vote is the **baseline without a language model**: its accuracy is what the LLM must beat. Compare it with the TF-IDF classifier of Session 13 (about 0.77 on the same years with 37,000 training decisions; here only 10,000 decisions are in the collection).

## 5. The RAG prototype — REQUIRES AN LLM ENDPOINT

In [ ]:
from openai import OpenAI

from bti_assistant.rag import suggest_heading
from bti_assistant.retrieval import VectorRetriever
from bti_assistant.store import NumpyVectorStore


class E5:                                             # the cached vectors, wrapped as an embedder
    dim = 384

    def encode(self, texts, kind="passage"):
        return model.encode([f"{kind}: {t}" for t in texts], normalize_embeddings=True)


store = NumpyVectorStore(384)
store.add(chunk_documents(dict(zip(refs, docs["description"])), size=10_000, overlap=0), E, list(docs["heading"]))
retriever = VectorRetriever(E5(), store)

BASE_URL = os.environ.get("LLM_BASE_URL", "http://localhost:11434/v1")
MODEL = os.environ.get("LLM_MODEL", "llama3.2")
client = OpenAI(base_url=BASE_URL, api_key=os.environ.get("LLM_API_KEY", "ollama"), timeout=120, max_retries=0)
try:
    client.models.list()
    LLM_OK = True
except Exception as e:
    LLM_OK = False
    print(f"No LLM reachable at {BASE_URL} ({type(e).__name__}): section 5 is skipped.")

In [ ]:
if LLM_OK:
    rows = []
    for _, r in ten.iterrows():
        s = suggest_heading(r["description"], retriever, client, MODEL, heading_texts, k=10)
        rows.append({"language": r["language"], "true": r["heading"], "proposed": s.heading,
                     "correct": s.heading == r["heading"], "in candidates": r["heading"] in s.candidates,
                     "citation_precision": s.citation_precision, "error": s.error, "reason": s.reason[:80]})
    answers = pd.DataFrame(rows)
    print(answers[["correct", "in candidates", "citation_precision"]].mean().round(2).to_dict())
    display(answers)
else:
    print("skipped: no LLM endpoint")

## 6. Rate the answers

Automatic checks cover correctness (the heading is known) and invented references. Groundedness and usefulness need a person. Fill in the table for the ten answers, ideally with two raters, and compute Cohen's kappa (`bti_assistant.metrics.cohen_kappa`).

| Request | grounded (0/1) | useful for an officer (1–5) | note |
|---|---|---|---|
| de | | | |
| fr | | | |
| ... | | | |

## Your turn

1. Which variant would you deploy, and why? Use the 500-request results, and state what the ten requests showed that the 500 could not.
2. For the requests where the true heading is not among the candidates, read the top decisions: is the collection missing such goods, or is retrieval failing?
3. Implement a multilingual re-ranker (for example `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`, about 470 MB) and compare it with the English one of the theory page.
4. With an LLM: is the assistant more accurate than the similarity vote? Report both on the same ten (better: 100) requests.

In [ ]:
# your code here